# BIRD Evaluation — Deterministic Schema-Linking Text-to-SQL

Runs the `Stop-Guessing-Joins` prototype against the **BIRD dev** set with a **local LLM** (Qwen2.5-Coder family — current open-weights SOTA for SQL).

Pipeline per question:
1. Open `dev_databases/<db_id>/<db_id>.sqlite`
2. Introspect schema → `Schema(tables, foreign_keys)`
3. Build `TextToSQLAgent` with that schema + the live SQLite connection
4. Run `agent.run(question + evidence)` → predicted SQL (local model, 4-bit)
5. Write predictions in BIRD's official format and run `evaluation.py` / `evaluation_ves.py`

Designed for **Google Colab**. `HF_TOKEN` is read from Colab Secrets (`google.colab.userdata`) — only needed for gated models; Qwen is open.

## 1. Setup — clone repo, install deps

Includes `transformers`, `accelerate`, `bitsandbytes` for 4-bit local inference.

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/mayursk2000/Stop-Guessing-Joins-Deterministic-Schema-Linking-for-Text-to-SQL-Agents.git"
REPO_DIR = pathlib.Path("/content/Stop-Guessing-Joins-Deterministic-Schema-Linking-for-Text-to-SQL-Agents")

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "sqlglot", "huggingface_hub", "requests", "tqdm",
    "transformers>=4.45", "accelerate>=0.34", "bitsandbytes>=0.43", "sentencepiece",
])
print("Repo:", REPO_DIR)

## 2. Secrets — pull `HF_TOKEN` from Colab Secrets

Add a secret named `HF_TOKEN` in the Colab sidebar (🔑 icon). Required only if you switch to a gated model (Llama, Gemma). Qwen2.5-Coder needs no token.

In [ ]:
HF_TOKEN = None
try:
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as e:
        print("HF_TOKEN not available:", e)
except ImportError:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

print("HF_TOKEN set:", bool(HF_TOKEN))

## 3. GPU check — pick a model size that fits

Defaults:
- **A100 80GB** → `Qwen2.5-Coder-32B-Instruct` in 4-bit (~18 GB), or BF16 (~64 GB) if you want max quality
- **A100 40GB / L4 24GB** → `Qwen2.5-Coder-32B-Instruct` 4-bit (~18 GB), comfortable
- **T4 16GB (free Colab)** → `Qwen2.5-Coder-14B-Instruct` 4-bit (~9 GB), or 7B if you want headroom
- **CPU only** → don't. Use the deterministic local generator (skip §5/§6 model load).

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name}  |  VRAM: {vram_gb:.1f} GB")
    if vram_gb >= 35:
        SUGGESTED_MODEL = "Qwen/Qwen2.5-Coder-32B-Instruct"
    elif vram_gb >= 20:
        SUGGESTED_MODEL = "Qwen/Qwen2.5-Coder-32B-Instruct"  # 4-bit fits in ~18GB
    elif vram_gb >= 14:
        SUGGESTED_MODEL = "Qwen/Qwen2.5-Coder-14B-Instruct"
    else:
        SUGGESTED_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
else:
    print("No GPU detected. Set Runtime → Change runtime type → GPU.")
    SUGGESTED_MODEL = None

print("Suggested model:", SUGGESTED_MODEL)

## 4. Download BIRD dev

Open **https://bird-bench.github.io/** in a browser, find the **Dev set** download button, **right-click → Copy link**, and paste *that* into `BIRD_DEV_URL` below.

The dev download is hosted on **Google Drive** (~3 GB). The cell below uses `gdown` to handle Drive's virus-scan interstitial automatically. Direct `https://...` URLs to a real zip also work.

> ⚠️ **Don't paste `https://bird-bench.github.io/` itself** — that's the HTML homepage, not the zip. If your downloaded file is a few hundred KB, that's what happened.

Expected layout after extraction:
```
/content/bird/dev_2024xx/
  dev.json
  dev_tables.json
  dev_databases/<db_id>/<db_id>.sqlite
```


In [ ]:
import re, zipfile, json, subprocess, sys, urllib.request

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
import gdown

BIRD_ROOT = pathlib.Path("/content/bird")
BIRD_ROOT.mkdir(exist_ok=True)


BIRD_DEV_URL = "https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip"

zip_path = BIRD_ROOT / "dev.zip"


def _looks_like_html(p: pathlib.Path) -> bool:
    if not p.exists() or p.stat().st_size < 1_000_000:  # real dev.zip is GBs
        try:
            head = p.read_bytes()[:512].lower()
            return b"<!doctype html" in head or b"<html" in head
        except Exception:
            return True
    return False


def _drive_id(url: str) -> str | None:
    for pat in (r"/file/d/([A-Za-z0-9_-]{20,})", r"[?&]id=([A-Za-z0-9_-]{20,})"):
        m = re.search(pat, url)
        if m:
            return m.group(1)
    return None


# Re-download if the existing file is bad (HTML / truncated)
if zip_path.exists() and _looks_like_html(zip_path):
    print(f"Existing {zip_path} looks like an HTML page ({zip_path.stat().st_size} bytes). Removing and re-downloading.")
    zip_path.unlink()

if BIRD_DEV_URL and not zip_path.exists():
    drive_id = _drive_id(BIRD_DEV_URL)
    if drive_id:
        print(f"Google Drive download (id={drive_id}) ...")
        gdown.download(id=drive_id, output=str(zip_path), quiet=False)
    else:
        print("Direct download ...")
        urllib.request.urlretrieve(BIRD_DEV_URL, zip_path)
    print(f"Downloaded {zip_path.stat().st_size:,} bytes -> {zip_path}")

# Hard-fail with a clear message rather than crashing inside zipfile
assert zip_path.exists(), "BIRD_DEV_URL is empty or download failed."
assert not _looks_like_html(zip_path), (
    f"{zip_path} is HTML ({zip_path.stat().st_size:,} bytes), not a zip. "
    "You probably pasted the bird-bench.github.io homepage. "
    "Open that page, right-click the 'Dev set' download button → Copy link, paste THAT into BIRD_DEV_URL, and re-run."
)

if not list(BIRD_ROOT.rglob("dev.json")):
    print("Extracting ...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(BIRD_ROOT)
    for inner in BIRD_ROOT.rglob("dev_databases.zip"):
        with zipfile.ZipFile(inner) as z:
            z.extractall(inner.parent)

candidates = [p.parent for p in BIRD_ROOT.rglob("dev.json")]
assert candidates, f"dev.json not found under {BIRD_ROOT} after extraction."
BIRD_DEV = candidates[0]
BIRD_DBS = next(BIRD_DEV.rglob("dev_databases"), BIRD_DEV / "dev_databases")
print("BIRD_DEV =", BIRD_DEV)
print("BIRD_DBS =", BIRD_DBS)
print("Questions:", len(json.loads((BIRD_DEV / "dev.json").read_text())))
print("Databases:", len(list(BIRD_DBS.iterdir())) if BIRD_DBS.exists() else "missing")


## 5. BIRD adapter — schema introspection + dev loader (BIRD-aware)

Builds a `prototype.Schema` enriched with BIRD's shipped metadata (used by every BIRD submission, so this is fair game — not test-set leakage):

- **Foreign keys** come from `dev_tables.json` (BIRD's canonical FK list). SQLite `PRAGMA foreign_key_list` returns *zero* FKs on most BIRD DBs, so without this the graph would have no edges.
- **Column descriptions** come from `<db_id>/database_description/*.csv` (BIRD's hand-written column docs). These flow into `Column.description`, so the existing `HybridRetriever`/`searchable_text()` will index them — no retriever changes needed.
- Tables and column types still come from `PRAGMA table_info` on the live SQLite.

The graph and retriever architecture from `prototype.py` is untouched; we're only feeding them better signal.

In [ ]:
import sqlite3, csv
from text2sql_agent_prototype.prototype import Schema, Table, Column, ForeignKey

# ---- One-time loaders (cached across calls) ----
_BIRD_TABLES_JSON: dict | None = None  # db_id -> table metadata block

def _load_dev_tables() -> dict:
    """Read BIRD's dev_tables.json once; cache by db_id."""
    global _BIRD_TABLES_JSON
    if _BIRD_TABLES_JSON is None:
        path = next(BIRD_DEV.rglob("dev_tables.json"), None)
        if path is None:
            print("⚠️  dev_tables.json not found — graph FKs will be empty.")
            _BIRD_TABLES_JSON = {}
        else:
            blocks = json.loads(path.read_text())
            _BIRD_TABLES_JSON = {b["db_id"]: b for b in blocks}
    return _BIRD_TABLES_JSON

def _read_column_descriptions(db_id: str) -> dict[str, dict[str, str]]:
    """Return {table_lower: {col_lower: description}} from BIRD's CSVs."""
    desc_dir = BIRD_DBS / db_id / "database_description"
    out: dict[str, dict[str, str]] = {}
    if not desc_dir.exists():
        return out
    for csv_path in desc_dir.glob("*.csv"):
        tbl = csv_path.stem.lower()
        col_descs: dict[str, str] = {}
        try:
            with open(csv_path, encoding="utf-8", errors="replace") as f:
                for row in csv.DictReader(f):
                    col = (row.get("original_column_name") or "").strip()
                    desc = (row.get("column_description") or "").strip()
                    val_desc = (row.get("value_description") or "").strip()
                    full = " ; ".join(filter(None, [desc, val_desc]))
                    if col:
                        col_descs[col.lower()] = full
        except Exception as exc:
            print(f"  warn: couldn't read {csv_path.name}: {exc}")
        out[tbl] = col_descs
    return out

# ---- Public adapter ----
def open_db(db_id: str) -> sqlite3.Connection:
    db_path = BIRD_DBS / db_id / f"{db_id}.sqlite"
    assert db_path.exists(), f"missing {db_path}"
    return sqlite3.connect(str(db_path))

def schema_from_bird(db_id: str) -> Schema:
    """Build a prototype.Schema using PRAGMA + dev_tables.json FKs + BIRD column docs."""
    conn = open_db(db_id)
    try:
        cur = conn.cursor()
        table_names = [r[0] for r in cur.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
        ).fetchall()]

        col_descs = _read_column_descriptions(db_id)
        bird_meta = _load_dev_tables().get(db_id)

        tables: dict[str, Table] = {}
        for tname in table_names:
            cols_info = cur.execute(f'PRAGMA table_info("{tname}")').fetchall()
            tbl_descs = col_descs.get(tname.lower(), {})
            columns = tuple(
                Column(
                    name=c[1],
                    description=" | ".join(filter(None, [
                        c[2].lower() if c[2] else "",
                        tbl_descs.get(c[1].lower(), ""),
                    ])),
                )
                for c in cols_info
            )
            # Table description: pull from BIRD if available
            tbl_desc = tname.replace("_", " ")
            if bird_meta and tname in bird_meta.get("table_names_original", []):
                idx = bird_meta["table_names_original"].index(tname)
                if idx < len(bird_meta.get("table_names", [])):
                    tbl_desc = bird_meta["table_names"][idx] or tbl_desc
            tables[tname] = Table(name=tname, columns=columns, description=tbl_desc)

        # Foreign keys: prefer BIRD's canonical list
        foreign_keys: list[ForeignKey] = []
        if bird_meta:
            names_orig = bird_meta["table_names_original"]
            cols_orig  = bird_meta["column_names_original"]   # [(table_idx, col_name), ...]
            for fk_pair in bird_meta.get("foreign_keys", []):
                l_idx, r_idx = fk_pair[0], fk_pair[1]
                l_tbl = names_orig[cols_orig[l_idx][0]]
                l_col = cols_orig[l_idx][1]
                r_tbl = names_orig[cols_orig[r_idx][0]]
                r_col = cols_orig[r_idx][1]
                if l_tbl in tables and r_tbl in tables:
                    foreign_keys.append(ForeignKey(
                        left_table=l_tbl, left_column=l_col,
                        right_table=r_tbl, right_column=r_col,
                    ))

        # Union in any FKs PRAGMA finds (rare on BIRD, but cheap insurance)
        seen = {(fk.left_table, fk.left_column, fk.right_table, fk.right_column)
                for fk in foreign_keys}
        for tname in table_names:
            for fk in cur.execute(f'PRAGMA foreign_key_list("{tname}")').fetchall():
                _, _, ref_table, from_col, to_col, *_ = fk
                if ref_table in tables and (tname, from_col, ref_table, to_col) not in seen:
                    foreign_keys.append(ForeignKey(
                        left_table=tname, left_column=from_col,
                        right_table=ref_table, right_column=to_col,
                    ))

        return Schema(tables=tables, foreign_keys=foreign_keys)
    finally:
        conn.close()

def load_dev(bird_dev_dir: pathlib.Path) -> list[dict]:
    return json.loads((bird_dev_dir / "dev.json").read_text())

# ---- Graph stats helper (used in smoke test + slice) ----
def graph_stats_str(trace, schema: Schema) -> str:
    rt  = ", ".join(trace.retrieval.candidate_tables) or "(none)"
    jt  = ", ".join(trace.join_plan.tables) or "(none)"
    jj  = "; ".join(j.join_condition() for j in trace.join_plan.joins) or "(no joins)"
    un  = ", ".join(trace.join_plan.unresolved_tables) or "(none)"
    return (
        f"  schema:     {len(schema.tables)} tables, {len(schema.foreign_keys)} FKs\n"
        f"  retrieval → {rt}\n"
        f"  graph     → {jt}\n"
        f"  joins     → {jj}\n"
        f"  unresolved→ {un}"
    )


# Sanity Check

In [ ]:
print("BIRD_DEV =", BIRD_DEV)
print("BIRD_DBS =", BIRD_DBS)
print(f"Found {len(list(BIRD_DBS.iterdir()))} databases:")
for p in sorted(BIRD_DBS.iterdir()):
    if p.is_dir():
        sqlite_file = p / f"{p.name}.sqlite"
        size_mb = sqlite_file.stat().st_size / 1e6 if sqlite_file.exists() else 0
        has_descs = (p / "database_description").exists()
        print(f"  {p.name:30s}  {size_mb:7.1f} MB   descriptions={'yes' if has_descs else 'NO'}")


## 5b. Hybrid retriever — BM25 + dense embeddings + RRF

Replaces the placeholder `HybridRetriever` from `prototype.py` (which was lexical-only, despite the name) with the actual hybrid retrieval the paper describes:

- **Column-level corpus.** Each column becomes one searchable document:
  `"Table frpm. Column Free Meal Count (K-12): Number of K-12 students eligible for free meals"`.
  Column-granular, not table-granular — that's how schema RAG actually works.
- **Lexical (BM25)** with CamelCase-aware tokenization so `AvgScrMath` indexes as `{avg, scr, math}`.
- **Dense (`BAAI/bge-small-en-v1.5`)** — 33 M-param sentence encoder, 384-dim, normalized embeddings, cosine similarity. Catches *"math score"* → `AvgScrMath` even when no surface token matches.
- **Fusion: Reciprocal Rank Fusion** (k = 60). Parameter-free — no thresholds to sweep on dev.
- **Aggregation**: per-table score = max RRF score across its columns. Top-k tables → candidate set T → graph as before.

The graph + rewriter layers are unchanged. Only the retriever — explicitly the placeholder the paper says to replace — is improved.

In [ ]:
import re as _re
from collections import defaultdict
import numpy as np

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "rank_bm25", "sentence-transformers"])

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from text2sql_agent_prototype.prototype import (
    HybridRetriever, RetrievalResult, RetrievalMatch, TextToSQLAgent,
)

# One encoder, shared across all DBs. Lives next to Qwen — ~150 MB extra GPU mem.
ENCODER_NAME = "BAAI/bge-small-en-v1.5"
print(f"Loading encoder {ENCODER_NAME} ...")
encoder = SentenceTransformer(
    ENCODER_NAME,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
_dim = (encoder.get_embedding_dimension()
        if hasattr(encoder, "get_embedding_dimension")
        else encoder.get_sentence_embedding_dimension())
print(f"Encoder loaded ({_dim}-dim).")


def _smart_split(text: str) -> list[str]:
    """Tokenize for BM25: split on non-alpha, then CamelCase, lowercase, drop short tokens."""
    out: list[str] = []
    for word in _re.findall(r"[A-Za-z0-9]+", text):
        for piece in _re.findall(r"[A-Z]+(?=[A-Z][a-z])|[A-Z]?[a-z]+|[A-Z]+|[0-9]+", word):
            t = piece.lower()
            if len(t) >= 2:
                out.append(t)
    return out


class BIRDHybridRetriever(HybridRetriever):
    """BM25 + dense, fused via Reciprocal Rank Fusion. Column-level corpus."""

    RRF_K = 60  # standard RRF constant — not tuned

    def __init__(self, schema: Schema, db_id: str, top_k: int = 6):
        self.schema = schema
        self.db_id = db_id
        self.top_k = top_k
        self.min_score = 0.0  # graph does the structural filtering, not us

        # ---- Build column-level corpus (one doc per column) ----
        self.docs: list[str] = []
        self.doc_tables: list[str] = []
        for tname, table in schema.tables.items():
            for c in table.columns:
                doc = (
                    f"Table {tname} ({table.description}). "
                    f"Column {c.name}: {c.description or ''}"
                )
                self.docs.append(doc)
                self.doc_tables.append(tname)

        # ---- BM25 index ----
        self.bm25 = BM25Okapi([_smart_split(d) for d in self.docs])

        # ---- Dense embeddings (one-time per db_id) ----
        self.dense = encoder.encode(
            self.docs,
            normalize_embeddings=True,
            show_progress_bar=False,
            convert_to_numpy=True,
        )

    def retrieve(self, query: str) -> RetrievalResult:
        if not self.docs:
            return RetrievalResult(query=query, candidate_tables=[], matches=[])

        # BM25 ranking over column docs
        bm25_scores = self.bm25.get_scores(_smart_split(query))
        bm25_rank = {int(i): r for r, i in enumerate(np.argsort(-bm25_scores))}

        # Dense ranking over column docs
        q_emb = encoder.encode(
            [query], normalize_embeddings=True,
            show_progress_bar=False, convert_to_numpy=True,
        )[0]
        dense_scores = self.dense @ q_emb
        dense_rank = {int(i): r for r, i in enumerate(np.argsort(-dense_scores))}

        # Reciprocal Rank Fusion
        rrf: dict[int, float] = {}
        K = self.RRF_K
        for i in range(len(self.docs)):
            rrf[i] = 1.0 / (K + bm25_rank[i]) + 1.0 / (K + dense_rank[i])

        # Aggregate to table: max RRF score across the table's columns
        table_score: dict[str, float] = {}
        for i, tname in enumerate(self.doc_tables):
            s = rrf[i]
            if tname not in table_score or s > table_score[tname]:
                table_score[tname] = s

        # Top-k tables
        ranked = sorted(table_score.items(), key=lambda kv: -kv[1])[: self.top_k]
        matches = [
            RetrievalMatch(
                table=t, lexical_score=0.0, semantic_score=0.0,
                score=float(s), matched_terms=[],
            )
            for t, s in ranked
        ]
        return RetrievalResult(
            query=query,
            candidate_tables=[m.table for m in matches],
            matches=matches,
        )


# Cache one retriever per db_id so we don't re-embed every question.
_RETRIEVER_CACHE: dict[str, BIRDHybridRetriever] = {}

def make_bird_agent(db_id: str):
    """Build a TextToSQLAgent with BIRD-aware schema + hybrid retriever + LLM gen.

    Requires §6 to have run (defines `local_generator`). Probe in this cell uses
    the retriever directly so it can validate §5b before §6 loads the LLM.
    """
    sch  = schema_from_bird(db_id)
    conn = open_db(db_id)
    if db_id not in _RETRIEVER_CACHE:
        _RETRIEVER_CACHE[db_id] = BIRDHybridRetriever(sch, db_id)
    # use_pruning=False because the prototype's prune heuristics are sales-schema specific.
    # Our retriever already returns a closed top-k set; the graph filters structurally.
    agent = TextToSQLAgent(sch, conn, use_llm=False, use_pruning=False)
    agent.retriever = _RETRIEVER_CACHE[db_id]
    agent.generator = local_generator   # noqa: F821 — defined in §6
    return agent, conn, sch


# ---- Probe: test retriever in isolation (does NOT need §6 / model load) ----
_probe_q   = load_dev(BIRD_DEV)[0]
_probe_db  = _probe_q["db_id"]
_probe_sch = schema_from_bird(_probe_db)
_probe_retr = BIRDHybridRetriever(_probe_sch, _probe_db)
_RETRIEVER_CACHE[_probe_db] = _probe_retr  # cache so §7/§8 reuse it

_probe_text = _probe_q["question"] + " " + (_probe_q.get("evidence") or "")
_probe_result = _probe_retr.retrieve(_probe_text)
print(f"\nProbe q (db={_probe_db}): {_probe_text[:120]}")
print(f"Top-{len(_probe_result.candidate_tables)} tables → "
      f"{', '.join(_probe_result.candidate_tables)}")


## 5c. Wire in the DFC SQL rewriter (paper's Reactive Layer)

The paper's reactive layer is the **DFC SQL rewriter** (Summers et al., CIDR 2026, ref [5]). The package ships at `<repo>/sql_rewriter/` — already importable because §1 put `<repo>` on `sys.path`.

The prototype's `DFCRewriterAdapter` defaults to a hardcoded Windows path that doesn't exist on Colab, but its `from sql_rewriter import SQLRewriter` falls through to the package on `sys.path` and loads correctly. The only missing piece is `duckdb`, which `sql_rewriter/rewriter.py` imports at module top.

This cell installs DuckDB, verifies the load, and smoke-tests `transform_query` on a sample SQLite query. After it runs, every `AliasNormalizingRewriter` constructed by `make_bird_agent` will route through DFC → graph-FK repair → alias normalizer.

> **Caveat noted in trace:** DFC parses/emits in DuckDB dialect, while BIRD executes against SQLite. sqlglot is mostly forgiving across dialects but watch the `Generated → Rewritten` diff in §7's smoke test for any breakage. If a dialect issue surfaces we'll address it in code, not by skipping DFC.

In [ ]:
import subprocess, sys, os

# 1) Install duckdb (DFC SQLRewriter imports it at module top).
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])

# 2) Point the prototype's DFC adapter at the local sql_rewriter package.
os.environ["DFC_SQL_REWRITER_PATH"] = str(REPO_DIR / "sql_rewriter")

# 3) Confirm the import path resolves to OUR copy of sql_rewriter.
import sql_rewriter as _dfc_pkg
print(f"DFC sql_rewriter loaded from: {_dfc_pkg.__file__}")

# 4) Build a fresh DFC adapter and verify it picks up the package.
from text2sql_agent_prototype.prototype import DFCRewriterAdapter
from sql_rewriter import DFCPolicy, Resolution

_adapter = DFCRewriterAdapter()
print(f"DFC adapter available: {_adapter.available}")
if _adapter.error:
    print(f"  load error: {_adapter.error}")

# 5) Smoke-test transform_query on a SQLite-flavored statement.
if _adapter.available:
    test_sql = (
        'SELECT T2.MailStreet FROM frpm AS T1 '
        'JOIN schools AS T2 ON T1.CDSCode = T2.CDSCode '
        'ORDER BY T1."FRPM Count (K-12)" DESC LIMIT 1'
    )
    transformed, notes = _adapter.transform(test_sql, policies=[])
    print("\n--- DFC transform_query smoke test ---")
    print("Input:    ", test_sql)
    print("Output:   ", transformed)
    print("Unchanged:", transformed.strip() == test_sql.strip())

# 6) Verify the policy.py patch — DFCPolicy now accepts FK-shaped constraints.
print("\n--- FK-shaped DFCPolicy construction test ---")
try:
    fk_policy = DFCPolicy(
        sources=["frpm", "schools"],
        constraint="frpm.CDSCode = schools.CDSCode",
        on_fail=Resolution.REMOVE,
        description="FK frpm.CDSCode → schools.CDSCode",
    )
    print(f"  ✓ {fk_policy.get_identifier()}")
except ValueError as e:
    print(f"  ✗ FK policy rejected: {e}")
    print("    (policy.py patch may not have applied — re-import sql_rewriter or restart kernel)")

# Counter-test: aggregate-shaped non-join constraint should still be rejected.
print("\n--- Negative test: unaggregated non-join constraint should still be rejected ---")
try:
    bad_policy = DFCPolicy(
        sources=["frpm"],
        constraint="frpm.CDSCode = 'X'",       # column-vs-literal, not a join
        on_fail=Resolution.REMOVE,
    )
    print(f"  ✗ Unexpectedly accepted: {bad_policy.get_identifier()}")
except ValueError as e:
    print(f"  ✓ Correctly rejected ({type(e).__name__}: {str(e)[:80]}...)")


## 5d. Alias normalizer — last reactive pass after DFC

The DFC rewriter handles policy-driven repairs. We layer one more deterministic AST pass on top to catch the canonical LLM bug DFC doesn't address — declaring `FROM frpm AS T1` and then writing `frpm.CDSCode` (SQLite errors with *"no such column"*).

`AliasNormalizingRewriter` chains:
```
SQL → DFC.transform_query        (paper's reactive layer)
    → _repair_join_predicates    (graph-FK enforcement, sqlglot AST)
    → normalize_aliases          (alias-vs-table-name fix, sqlglot AST)
```

`make_bird_agent` is re-defined here to wire this rewriter into every agent.

In [ ]:
import sqlglot
from sqlglot import exp
from text2sql_agent_prototype.prototype import (
    SQLRewriter as _BaseSQLRewriter, RewriteResult,
)
from sql_rewriter import DFCPolicy, Resolution


def normalize_aliases(sql: str) -> tuple[str, list[str]]:
    """AST pass: rewrite column refs that use a real table name when the table
    has been aliased in the FROM/JOIN. Idempotent — no-ops on correct SQL.
    """
    notes: list[str] = []
    try:
        tree = sqlglot.parse_one(sql, dialect="sqlite")
    except Exception as exc:
        return sql, [f"alias-norm: parse failed ({exc}); passing through"]

    table_to_alias: dict[str, str] = {}
    for t in tree.find_all(exp.Table):
        if t.alias and t.name:
            table_to_alias[t.name.lower()] = t.alias

    if not table_to_alias:
        return sql, []

    n_rewrites = 0
    for col in tree.find_all(exp.Column):
        qualifier = col.table
        if not qualifier:
            continue
        new_alias = table_to_alias.get(qualifier.lower())
        if new_alias and qualifier != new_alias:
            col.set("table", exp.to_identifier(new_alias))
            n_rewrites += 1

    if n_rewrites == 0:
        return sql, []

    notes.append(f"alias-norm: rewrote {n_rewrites} column reference(s) to use table aliases")
    return tree.sql(dialect="sqlite"), notes


class AliasNormalizingRewriter(_BaseSQLRewriter):
    """Reactive layer: DFC.transform_query → graph-FK repair → alias normalization."""
    def rewrite(self, sql, join_plan):
        base = super().rewrite(sql, join_plan)
        new_sql, alias_notes = normalize_aliases(base.sql)
        if not alias_notes:
            return base
        return RewriteResult(
            sql=new_sql,
            changed=base.changed or new_sql != base.sql,
            notes=base.notes + alias_notes,
        )


def _seed_dfc_schema(dfc_rewriter, schema: Schema) -> None:
    """Create empty stub tables in the DFC rewriter's DuckDB so register_policy
    can validate FK constraints. We never query these stubs — they exist only
    so column-existence checks during policy registration pass.
    """
    for tname, table in schema.tables.items():
        cols_def = ", ".join(f'"{c.name}" VARCHAR' for c in table.columns)
        try:
            dfc_rewriter.conn.execute(f'CREATE TABLE IF NOT EXISTS "{tname}" ({cols_def})')
        except Exception as exc:
            print(f"  warn: couldn't create stub table {tname}: {exc}")


def _register_fk_policies(dfc_rewriter, schema: Schema) -> tuple[int, int]:
    """Register each schema.foreign_keys entry as an FK-shaped DFCPolicy.
    Returns (registered, attempted). Silent on individual failures.
    """
    registered = 0
    for fk in schema.foreign_keys:
        try:
            policy = DFCPolicy(
                sources=[fk.left_table, fk.right_table],
                constraint=(
                    f'"{fk.left_table}"."{fk.left_column}" = '
                    f'"{fk.right_table}"."{fk.right_column}"'
                ),
                on_fail=Resolution.REMOVE,
                description=f"FK: {fk.join_condition()}",
            )
            dfc_rewriter.register_policy(policy)
            registered += 1
        except Exception:
            pass    # silent; happens when stub-validation rejects an edge case
    return registered, len(schema.foreign_keys)


def make_bird_agent(db_id: str):
    """Build a TextToSQLAgent: BIRD schema + hybrid retriever + DFC + alias-fix + LLM gen."""
    sch  = schema_from_bird(db_id)
    conn = open_db(db_id)
    if db_id not in _RETRIEVER_CACHE:
        _RETRIEVER_CACHE[db_id] = BIRDHybridRetriever(sch, db_id)

    agent = TextToSQLAgent(sch, conn, use_llm=False, use_pruning=False)
    agent.retriever = _RETRIEVER_CACHE[db_id]
    agent.rewriter  = AliasNormalizingRewriter()
    agent.generator = local_generator   # noqa: F821 — defined in §6

    # Register FK constraints as DFCPolicies on the agent's DFC rewriter.
    if agent.rewriter.dfc_adapter.available:
        dfc = agent.rewriter.dfc_adapter._rewriter
        _seed_dfc_schema(dfc, sch)
        registered, attempted = _register_fk_policies(dfc, sch)
        if attempted:
            print(f"  [{db_id}] DFC FK policies: {registered}/{attempted} registered")

    return agent, conn, sch


# ---- Smoke-test the alias normalizer ----
_test_bad = (
    'SELECT * FROM frpm AS T1 JOIN schools AS T2 ON frpm.CDSCode = schools.CDSCode '
    'WHERE T1."County Name" = \'Alameda\''
)
_test_fixed, _notes = normalize_aliases(_test_bad)
print("Before:", _test_bad)
print("After: ", _test_fixed)
print("Notes: ", _notes)

_test_ok = (
    'SELECT * FROM frpm AS T1 JOIN schools AS T2 ON T1.CDSCode = T2.CDSCode '
    'WHERE T1."County Name" = \'Alameda\''
)
_test_ok2, _notes2 = normalize_aliases(_test_ok)
print("\nIdempotence: ", "OK" if _test_ok2 == _test_ok and not _notes2 else "FAILED")


## 6. Local LLM generator (Qwen2.5-Coder, 4-bit)

Loads the model **once** with `bitsandbytes` 4-bit NF4 quantization. Reused across all 1534 questions. Drop-in subclass of `SQLGenerator` — falls back to the deterministic local generator on any error.

**Model rationale:** Qwen2.5-Coder-32B-Instruct is the current open-weights SOTA for code/SQL on benchmarks like HumanEval, MBPP, BIRD itself. It generally beats Llama-3.1-70B on coding while being half the size and fully open (no license gate).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from text2sql_agent_prototype.prototype import SQLGenerator, GeneratedSQL, JoinPlan
import re

MODEL_NAME = SUGGESTED_MODEL or "Qwen/Qwen2.5-Coder-7B-Instruct"
USE_4BIT  = True   # set False on A100 80GB if you want full BF16
MAX_NEW_TOKENS = 384

print(f"Loading {MODEL_NAME} (4-bit={USE_4BIT}) ...")

_bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
) if USE_4BIT else None

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=_bnb_cfg,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
print("Loaded. Device map:",
      model.hf_device_map if hasattr(model, "hf_device_map") else "single device")

_FENCE = re.compile(r"^```(?:sql|sqlite)?\s*|\s*```$", re.IGNORECASE | re.MULTILINE)


def linked_schema_prompt(join_plan: JoinPlan, schema: Schema) -> str:
    """Render ONLY graph-resolved tables + their columns + BIRD descriptions.

    Paper's contribution: model sees only the schema-linked subset, never the
    whole DB. If the graph picked the wrong tables, this is where it shows up.
    """
    if not join_plan.tables:
        return "Linked schema: (graph produced no tables — retrieval likely failed)"

    blocks: list[str] = []
    for tname in join_plan.tables:
        table = schema.tables[tname]
        col_lines = []
        for c in table.columns:
            desc = c.description or ""
            col_lines.append(f'    "{c.name}"  -- {desc}' if desc else f'    "{c.name}"')
        blocks.append(f'Table "{tname}":\n' + "\n".join(col_lines))

    join_lines = [f"  {j.join_condition()}" for j in join_plan.joins] or [
        "  (no joins required — single table)"
    ]
    return (
        "Linked schema (graph-resolved tables and their columns):\n"
        + "\n\n".join(blocks)
        + "\n\nApproved joins (use ONLY these to connect tables):\n"
        + "\n".join(join_lines)
    )


class LocalHFGenerator(SQLGenerator):
    """Drop-in SQL generator using a locally-loaded HF model + linked-schema prompt."""
    def __init__(self, tokenizer, model, max_new_tokens: int = 384):
        self.tokenizer = tokenizer
        self.model = model
        self.max_new_tokens = max_new_tokens
        self.fallback = SQLGenerator()

    def _build_messages(self, query: str, join_plan: JoinPlan, schema: Schema) -> list[dict]:
        return [
            {"role": "system", "content":
                "You are an expert Text-to-SQL assistant for SQLite. "
                "You will be given the linked schema selected by a deterministic schema-linker: "
                "use ONLY the tables and columns shown. Do not invent column names. "
                "Use ONLY the approved joins listed; do not invent join predicates. "
                "Return EXACTLY ONE SQLite SELECT statement. No prose, no code fences. "
                "Quote column names containing spaces or special characters with double quotes."},
            {"role": "user", "content":
                f"{linked_schema_prompt(join_plan, schema)}\n\n"
                f"Question: {query}\n\n"
                "Output only the SELECT statement."},
        ]

    def _clean(self, text: str) -> str:
        text = _FENCE.sub("", text).strip()
        m = re.search(r"\b(SELECT|WITH)\b", text, re.IGNORECASE)
        if m:
            text = text[m.start():]
        return text.strip().rstrip(";").strip()

    def generate(self, query: str, join_plan: JoinPlan, schema: Schema) -> GeneratedSQL:
        try:
            msgs = self._build_messages(query, join_plan, schema)
            prompt = self.tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True,
            )
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
            with torch.inference_mode():
                out = self.model.generate(
                    **inputs,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=False,
                    pad_token_id=self.tokenizer.pad_token_id,
                )
            new_tokens = out[0, inputs.input_ids.shape[1]:]
            raw = self.tokenizer.decode(new_tokens, skip_special_tokens=True)
            sql = self._clean(raw)
            if not re.match(r"(?i)^\s*(select|with)\b", sql):
                raise ValueError(f"Non-SELECT output: {sql[:120]}")
            return GeneratedSQL(sql=sql, rationale=f"Local HF model {MODEL_NAME}")
        except Exception as exc:
            fb = self.fallback.generate(query, join_plan, schema)
            return GeneratedSQL(sql=fb.sql, rationale=f"Local model failed ({exc}); fallback used.")

local_generator = LocalHFGenerator(tokenizer, model, max_new_tokens=MAX_NEW_TOKENS)
print("LocalHFGenerator ready (linked-schema prompt — graph-constrained).")


## 7. Smoke test — run the agent end-to-end on one BIRD question

Sanity check before burning 20+ LLM calls in §8: open Q0, run retrieval → graph → LLM → rewriter → exec → validation, print the trace. If the graph picks at least one of the gold tables and the generated SQL is a SELECT, plumbing's healthy.

In [ ]:
from text2sql_agent_prototype.prototype import TextToSQLAgent

# Evidence appears verbatim in the question's reference SQL ~half the time.
# We pass it to retrieval AND to the LLM (the LLM still sees evidence as part of nl).
def build_retrieval_query(q: dict) -> str:
    parts = [q["question"]]
    if q.get("evidence"):
        parts.append(q["evidence"])
    return " ".join(parts)

def build_llm_query(q: dict) -> str:
    return q["question"] + ((" " + q["evidence"]) if q.get("evidence") else "")

questions = load_dev(BIRD_DEV)
q0 = questions[0]
print("db_id:    ", q0["db_id"])
print("question: ", q0["question"])
print("evidence: ", q0.get("evidence", ""))
print("gold SQL: ", q0["SQL"])
print("-" * 80)

agent, conn, schema = make_bird_agent(q0["db_id"])
print(f"Schema: {len(schema.tables)} tables, {len(schema.foreign_keys)} FKs")

retrieval_q = build_retrieval_query(q0)
trace = agent.run(retrieval_q)

print("\n--- Graph stats ---")
print(graph_stats_str(trace, schema))

print("\nGenerated:", trace.generated.sql)
print("Rewritten:", trace.rewrite.sql)
print("Validation:", trace.validation.valid, "-", trace.validation.reason)
print("Final SQL:", trace.final_sql or "(falling back to rewrite.sql)")
conn.close()


## 8. Run on a slice → predictions in BIRD format

BIRD's `evaluation.py` expects a JSON mapping `"<question_id>"` → `"<SQL>\t----- bird -----\t<db_id>"`. Predictions are checkpointed every 25 questions; if the cell crashes, just re-run — it resumes from disk. Set `N = len(questions)` for full dev. Per-question graph stats land in `/content/graph_stats.jsonl` for §8b's diagnosis.

In [ ]:
from tqdm.auto import tqdm

N = 20  # set to len(questions) for full dev (~1534)
PRED_PATH       = pathlib.Path("/content/predict_dev.json")
GRAPH_LOG_PATH  = pathlib.Path("/content/graph_stats.jsonl")

predictions: dict[str, str] = {}
if PRED_PATH.exists():
    predictions = json.loads(PRED_PATH.read_text())
    print(f"Resuming with {len(predictions)} predictions already on disk")

agent_cache:  dict[str, TextToSQLAgent]   = {}
schema_cache: dict[str, Schema]            = {}
conn_cache:   dict[str, sqlite3.Connection] = {}

def get_agent(db_id: str) -> TextToSQLAgent:
    if db_id not in agent_cache:
        agent, conn, sch = make_bird_agent(db_id)
        agent_cache[db_id]  = agent
        conn_cache[db_id]   = conn
        schema_cache[db_id] = sch
    return agent_cache[db_id]

graph_lines: list[dict] = []

for i, q in enumerate(tqdm(questions[:N])):
    qid = str(q.get("question_id", i))
    if qid in predictions:
        continue
    retrieval_q = build_retrieval_query(q)
    try:
        agent = get_agent(q["db_id"])
        trace = agent.run(retrieval_q)
        sql = (trace.final_sql or trace.rewrite.sql or "SELECT 1").replace("\n", " ").strip()
        graph_lines.append({
            "qid": qid,
            "db_id": q["db_id"],
            "retrieval": list(trace.retrieval.candidate_tables),
            "graph_tables": list(trace.join_plan.tables),
            "graph_joins": [j.join_condition() for j in trace.join_plan.joins],
            "unresolved": list(trace.join_plan.unresolved_tables),
            "schema_size": (
                len(schema_cache[q["db_id"]].tables),
                len(schema_cache[q["db_id"]].foreign_keys),
            ),
        })
    except Exception as exc:
        sql = "SELECT 1"
        print(f"[{qid}] error:", exc)
        graph_lines.append({"qid": qid, "db_id": q["db_id"], "error": str(exc)})
    predictions[qid] = f"{sql}\t----- bird -----\t{q['db_id']}"
    if (i + 1) % 25 == 0:
        PRED_PATH.write_text(json.dumps(predictions, indent=2))

for c in conn_cache.values():
    c.close()
agent_cache.clear()

PRED_PATH.write_text(json.dumps(predictions, indent=2))
GRAPH_LOG_PATH.write_text("\n".join(json.dumps(g) for g in graph_lines))
print(f"\nWrote {PRED_PATH} ({len(predictions)} predictions)")
print(f"Wrote {GRAPH_LOG_PATH} ({len(graph_lines)} graph traces)")

print("\n--- Graph stats for first 5 questions ---")
for g in graph_lines[:5]:
    if "error" in g:
        print(f"\n  qid={g['qid']} db={g['db_id']}  ERROR: {g['error']}")
        continue
    print(f"\n  qid={g['qid']} db={g['db_id']}  "
          f"schema={g['schema_size'][0]}t/{g['schema_size'][1]}fk")
    print(f"    retrieval → {', '.join(g['retrieval']) or '(none)'}")
    print(f"    graph     → {', '.join(g['graph_tables']) or '(none)'}")
    print(f"    joins     → {'; '.join(g['graph_joins']) or '(no joins)'}")
    if g["unresolved"]:
        print(f"    unresolved→ {', '.join(g['unresolved'])}")


## 8b. Local metrics — Join Accuracy + Execution Accuracy

Reproduces the paper's reporting format (Table 1 of the abstract):

| Method            | Join Acc. | Exec. Acc. |
|-------------------|----------:|-----------:|
| Prior Work [BIRD] | 58.2      | 54.9       |
| Agent-based       | 63.7      | 57.1       |
| **Ours (target)** | **73.8**  | **64.4**   |

- **Join Accuracy** — average F1 between the set of join predicates in predicted vs gold SQL, averaged over multi-table questions only. Each join is a `{table.col, table.col}` set, parsed by `sqlglot`. Single-table queries are excluded (no joins to score).
- **Execution Accuracy (EX)** — predicted SQL produces the same result rows as gold SQL on the BIRD SQLite DB. Approximate local version of BIRD's official `evaluation.py`.

In [ ]:
import sqlite3, json
from collections import defaultdict
import sqlglot
from sqlglot import exp

PRED_PATH      = pathlib.Path("/content/predict_dev.json")
GRAPH_LOG_PATH = pathlib.Path("/content/graph_stats.jsonl")

predictions = json.loads(PRED_PATH.read_text())
questions   = load_dev(BIRD_DEV)
qmap = {str(q.get("question_id", i)): q for i, q in enumerate(questions)}
graph_log = {g["qid"]: g for g in (json.loads(l) for l in GRAPH_LOG_PATH.read_text().splitlines() if l.strip())} \
            if GRAPH_LOG_PATH.exists() else {}

TIMEOUT_S = 10
_NULL = "\x00NULL\x00"

# -------- Execution Accuracy --------
def run_sql(db_id: str, sql: str):
    db_path = BIRD_DBS / db_id / f"{db_id}.sqlite"
    conn = sqlite3.connect(str(db_path), timeout=TIMEOUT_S)
    try:
        conn.execute(f"PRAGMA busy_timeout = {TIMEOUT_S * 1000}")
        return conn.execute(sql).fetchall(), None
    except Exception as exc:
        return None, str(exc)
    finally:
        conn.close()

def normalize(rows):
    if rows is None:
        return None
    return sorted(tuple(_NULL if c is None else str(c) for c in r) for r in rows)

# -------- Join Accuracy --------
def extract_joins(sql: str) -> set[frozenset[str]]:
    try:
        tree = sqlglot.parse_one(sql, dialect="sqlite")
    except Exception:
        return set()
    alias_map: dict[str, str] = {}
    for t in tree.find_all(exp.Table):
        name = (t.name or "").lower()
        alias = (t.alias or t.name or "").lower()
        if name and alias:
            alias_map[alias] = name
    joins: set[frozenset[str]] = set()
    for eq in tree.find_all(exp.EQ):
        left, right = eq.this, eq.expression
        if isinstance(left, exp.Column) and isinstance(right, exp.Column):
            lt = (left.table or "").lower()
            rt = (right.table or "").lower()
            if lt and rt and lt in alias_map and rt in alias_map and lt != rt:
                a = f"{alias_map[lt]}.{left.name.lower()}"
                b = f"{alias_map[rt]}.{right.name.lower()}"
                joins.add(frozenset([a, b]))
    return joins

def join_f1(pred_sql: str, gold_sql: str):
    gold = extract_joins(gold_sql)
    if not gold:
        return None
    pred = extract_joins(pred_sql)
    if not pred:
        return 0.0
    tp = len(pred & gold)
    if tp == 0:
        return 0.0
    p = tp / len(pred)
    r = tp / len(gold)
    return 2 * p * r / (p + r)

def gold_tables(sql: str) -> set[str]:
    try:
        tree = sqlglot.parse_one(sql, dialect="sqlite")
    except Exception:
        return set()
    return {(t.name or "").lower() for t in tree.find_all(exp.Table) if t.name}

# -------- Score --------
results = []
for qid, line in predictions.items():
    if qid not in qmap:
        continue
    q = qmap[qid]
    pred_sql = line.split("\t----- bird -----\t")[0].strip()
    is_fallback = pred_sql.rstrip(";").strip().upper() == "SELECT 1"

    pred_rows, pred_err = run_sql(q["db_id"], pred_sql)
    gold_rows, gold_err = run_sql(q["db_id"], q["SQL"])

    if pred_err:        ex = "exec_error"
    elif gold_err:      ex = "gold_error"
    elif normalize(pred_rows) == normalize(gold_rows): ex = "correct"
    else:               ex = "wrong_result"

    # Graph table-recall: did the graph pick the tables gold actually needs?
    g_info = graph_log.get(qid, {})
    graph_set = {t.lower() for t in g_info.get("graph_tables", [])}
    gold_set  = gold_tables(q["SQL"])
    graph_tbl_recall = (len(graph_set & gold_set) / len(gold_set)) if gold_set else None

    results.append({
        "qid": qid, "db_id": q["db_id"], "difficulty": q.get("difficulty", "?"),
        "ex": ex, "pred_err": pred_err, "fallback": is_fallback,
        "jf1": join_f1(pred_sql, q["SQL"]), "pred_sql": pred_sql,
        "graph_info": g_info, "graph_tbl_recall": graph_tbl_recall,
    })

# -------- Headline --------
total      = len(results)
ex_correct = sum(r["ex"] == "correct" for r in results)
ex_acc     = ex_correct / max(total, 1) * 100
scored_join = [r for r in results if r["jf1"] is not None]
join_acc    = (sum(r["jf1"] for r in scored_join) / max(len(scored_join), 1)) * 100
scored_recall = [r for r in results if r["graph_tbl_recall"] is not None]
graph_recall  = (sum(r["graph_tbl_recall"] for r in scored_recall) / max(len(scored_recall), 1)) * 100

print(f"=== Local metrics on {total} predictions ===\n")
print(f"{'Method':<22s}{'Join Acc.':>12s}{'Exec. Acc.':>13s}")
print(f"{'-'*47}")
print(f"{'Prior Work [BIRD]':<22s}{58.2:>12.1f}{54.9:>13.1f}")
print(f"{'Agent-based':<22s}{63.7:>12.1f}{57.1:>13.1f}")
print(f"{'Ours (paper target)':<22s}{73.8:>12.1f}{64.4:>13.1f}")
print(f"{'This run':<22s}{join_acc:>12.1f}{ex_acc:>13.1f}")

print(f"\nUpstream signal:")
print(f"  Graph-table recall: {graph_recall:.1f}%   (did the graph include every gold table?)")
print(f"  Join Acc. averaged over {len(scored_join)} multi-table questions; {total - len(scored_join)} single-table excluded.")

print(f"\nEX status:  correct={ex_correct}  "
      f"wrong={sum(r['ex']=='wrong_result' for r in results)}  "
      f"err={sum(r['ex']=='exec_error' for r in results)}  "
      f"fallback={sum(r['fallback'] for r in results)}")

print(f"\n--- By difficulty ---")
print(f"{'difficulty':<14s}{'Join Acc.':>11s}{'Exec. Acc.':>12s}{'cases':>8s}")
for d in ("simple", "moderate", "challenging", "?"):
    bucket = [r for r in results if r["difficulty"] == d]
    if not bucket:
        continue
    bj = [r for r in bucket if r["jf1"] is not None]
    j = (sum(r["jf1"] for r in bj) / max(len(bj), 1)) * 100 if bj else float("nan")
    e = sum(r["ex"] == "correct" for r in bucket) / len(bucket) * 100
    print(f"  {d:<12s}{j:>11.1f}{e:>12.1f}{len(bucket):>8d}")

print(f"\n--- By db_id ---")
print(f"{'db_id':<26s}{'Join Acc.':>11s}{'Exec. Acc.':>12s}{'cases':>8s}")
by_db: dict[str, list] = defaultdict(list)
for r in results:
    by_db[r["db_id"]].append(r)
for d, bucket in sorted(by_db.items()):
    bj = [r for r in bucket if r["jf1"] is not None]
    j = (sum(r["jf1"] for r in bj) / max(len(bj), 1)) * 100 if bj else float("nan")
    e = sum(r["ex"] == "correct" for r in bucket) / len(bucket) * 100
    print(f"  {d:<24s}{j:>11.1f}{e:>12.1f}{len(bucket):>8d}")

# -------- Failures with graph diagnosis --------
print("\n--- First 5 EX failures (with graph diagnosis) ---")
shown = 0
for r in results:
    if r["ex"] == "correct" or shown >= 5:
        continue
    q = qmap[r["qid"]]
    jf = "—" if r["jf1"] is None else f"{r['jf1']:.2f}"
    g  = r["graph_info"]
    gold_set = gold_tables(q["SQL"])
    graph_set = {t.lower() for t in g.get("graph_tables", [])}
    missing = gold_set - graph_set
    extra   = graph_set - gold_set
    print(f"\n  qid={r['qid']} ({r['difficulty']}) db={r['db_id']}  ex={r['ex']}  join_f1={jf}")
    print(f"    Q:    {q['question'][:140]}")
    if q.get("evidence"):
        print(f"    ev:   {q['evidence'][:140]}")
    print(f"    retrieval → {', '.join(g.get('retrieval', [])) or '(none)'}")
    print(f"    graph     → {', '.join(g.get('graph_tables', [])) or '(none)'}")
    print(f"    gold tabs → {', '.join(sorted(gold_set)) or '(none)'}")
    if missing:
        print(f"    *** graph MISSED tables: {', '.join(sorted(missing))}")
    if extra:
        print(f"    *** graph added EXTRA tables: {', '.join(sorted(extra))}")
    print(f"    pred: {r['pred_sql'][:160]}")
    print(f"    gold: {q['SQL'][:160]}")
    if r["pred_err"]:
        print(f"    err:  {r['pred_err'][:160]}")
    shown += 1


## 9. Run BIRD's official evaluation scripts

Drop `evaluation.py` and `evaluation_ves.py` from the BIRD repo's `evaluation/` folder into `/content/bird_eval/`. The cell below builds the gold-SQL file and runs both EX and VES.

In [ ]:
BIRD_EVAL = pathlib.Path("/content/bird_eval")
BIRD_EVAL.mkdir(exist_ok=True)

GT_PATH = BIRD_EVAL / "dev_gold.sql"
lines = []
for q in questions:
    sql = q["SQL"].replace("\n", " ").strip()
    lines.append(f"{sql}\t{q['db_id']}")
GT_PATH.write_text("\n".join(lines))
print("Wrote", GT_PATH)

ex_script  = BIRD_EVAL / "evaluation.py"
ves_script = BIRD_EVAL / "evaluation_ves.py"

if ex_script.exists():
    !python {ex_script} \
        --predicted_sql_path {PRED_PATH} \
        --ground_truth_path {GT_PATH} \
        --db_root_path {BIRD_DBS} \
        --num_cpus 4 \
        --meta_time_out 30.0 \
        --mode_gt gt --mode_predict gpt
else:
    print(f"Place evaluation.py at {ex_script} and re-run.")

if ves_script.exists():
    !python {ves_script} \
        --predicted_sql_path {PRED_PATH} \
        --ground_truth_path {GT_PATH} \
        --db_root_path {BIRD_DBS} \
        --num_cpus 4 \
        --meta_time_out 30.0 \
        --mode_gt gt --mode_predict gpt
else:
    print(f"Place evaluation_ves.py at {ves_script} and re-run.")

## Notes & next steps

**Throughput.** With 4-bit Qwen2.5-Coder-32B on an A100 40GB, expect ~2–4 sec/question greedy → ~1–2 hours for full dev (1534 Q). 7B is ~3× faster. For real production-scale eval, swap `transformers.generate` for **vLLM** (`pip install vllm`) — usually 5–10× speedup via continuous batching.

**Ablations to run for the poster narrative:**
1. **Local model + graph (this notebook)** → main result.
2. **Local model, no graph constraints** → re-run with `agent.use_pruning = False` *and* feed the model the raw retrieval candidates instead of `join_plan.prompt_context`. Comparable to the README's "Agent-based" row.
3. **Deterministic local generator** → set `agent.generator = SQLGenerator()`. Sanity check that the plumbing works without an LLM.

The poster's contribution is the **delta in EX** between (1) and (2).

**Eval CLI flags** vary across BIRD releases. If a flag is rejected, run `python evaluation.py --help` and adjust.

**DFC rewriter** (the Windows-path-bound sibling project in `prototype.py`) is unavailable on Colab — it falls back silently and graph-policy notes still appear in the trace.

**Other model swaps** worth trying:
- `defog/sqlcoder-70b-alpha` — SQL-specialized, but 2023-vintage; expect lower BIRD scores than Qwen2.5-Coder-32B.
- `meta-llama/Meta-Llama-3.1-70B-Instruct` — strong general; needs HF_TOKEN + license accept; ~40 GB in 4-bit.
- `deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct` — 16B MoE (2.4B active), fast and competitive on code.